# Predict Future Stock Prices (Short-Term)

**Objective:** Use historical stock data to predict the next day's closing price using regression models.

In [ ]:
# ── Imports ──────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
print("All libraries loaded!")

## 1. Fetch Stock Data

In [ ]:
# Download Apple (AAPL) historical data — last 3 years
TICKER = "AAPL"
stock = yf.Ticker(TICKER)
df = stock.history(period="3y")

print(f"Stock: {TICKER}")
print(f"Shape: {df.shape}")
print(f"Date range: {df.index[0].date()} → {df.index[-1].date()}")
df.head()

## 2. Feature Engineering

In [ ]:
# Keep relevant columns
df = df[['Open','High','Low','Close','Volume']].copy()
df.dropna(inplace=True)

# Target: next day's Close (shift by -1)
df['Target'] = df['Close'].shift(-1)

# Additional features
df['Price_Range']   = df['High'] - df['Low']
df['Close_OpenDiff']= df['Close'] - df['Open']
df['MA_5']          = df['Close'].rolling(5).mean()
df['MA_20']         = df['Close'].rolling(20).mean()
df['Volatility']    = df['Close'].rolling(5).std()

df.dropna(inplace=True)
print(f"Final dataset shape: {df.shape}")
df.head()

## 3. Train / Test Split

In [ ]:
FEATURES = ['Open','High','Low','Volume','Price_Range',
            'Close_OpenDiff','MA_5','MA_20','Volatility']
TARGET = 'Target'

X = df[FEATURES]
y = df[TARGET]

# Chronological split — 80% train, 20% test
split = int(len(df) * 0.80)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

# Scale features
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")

## 4. Train Models

In [ ]:
# ── Linear Regression ────────────────────────────────────
lr = LinearRegression()
lr.fit(X_train_s, y_train)
lr_preds = lr.predict(X_test_s)

# ── Random Forest Regressor ───────────────────────────────
rf = RandomForestRegressor(n_estimators=200, max_depth=10,
                           random_state=42, n_jobs=-1)
rf.fit(X_train_s, y_train)
rf_preds = rf.predict(X_test_s)

def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"{name:25s} | MAE: ${mae:.2f}  RMSE: ${rmse:.2f}  R²: {r2:.4f}")

print(f"{'Model':25s} | {'MAE':10s}  {'RMSE':10s}  {'R²':8s}")
print("-" * 60)
evaluate("Linear Regression",  y_test, lr_preds)
evaluate("Random Forest",       y_test, rf_preds)

## 5. Visualize Actual vs Predicted

In [ ]:
dates = df.index[split:]
fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

for ax, preds, name, color in zip(
        axes,
        [lr_preds, rf_preds],
        ["Linear Regression", "Random Forest"],
        ["#e74c3c", "#27ae60"]):
    ax.plot(dates, y_test.values, label='Actual Close', color='#2c3e50', lw=1.5)
    ax.plot(dates, preds,         label=f'{name} Predicted', color=color, lw=1.5, alpha=0.8)
    ax.set_title(f'{TICKER} — {name}: Actual vs Predicted', fontweight='bold', fontsize=12)
    ax.legend()
    ax.set_ylabel('Price (USD)')

axes[-1].set_xlabel('Date')
plt.suptitle(f'{TICKER} Stock Price Prediction', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('task2_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved.")

## 6. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(9, 5))
importances.plot(kind='barh', color=sns.color_palette('Set2', len(FEATURES)))
plt.title('Random Forest — Feature Importances', fontweight='bold', fontsize=13)
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('task2_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Key Insights & Findings

- **Random Forest** significantly outperforms Linear Regression (higher R², lower MAE/RMSE).
- **Moving averages (MA_5, MA_20)** are the most important features, confirming momentum effects.
- The next-day close prediction is very close to actual prices, showing the model captures short-term trends well.
- For production use, additional features (news sentiment, technical indicators) would improve performance further.
